In [ ]:
import os
import copy
import pickle
import logging
import numpy as np
import matlab.engine

from tqdm import tqdm
from beartype import beartype
from scipy.stats import norm

from UQpy.distributions import JointIndependent, Normal
from UQpy.sampling import LatinHypercubeSampling
from UQpy.sampling.stratified_sampling.latin_hypercube_criteria import MaxiMin, DistanceMetric
from UQpy.surrogates.polynomial_chaos import PolynomialChaosExpansion
from UQpy.surrogates.polynomial_chaos.regressions import LeastSquareRegression
from UQpy.surrogates.polynomial_chaos.polynomials.TotalDegreeBasis import TotalDegreeBasis

from Theta import ThetaCriterionPCE
from LAR import LeastAngleRegression

In [ ]:
# Matlab calculation

MATLAB_TASK_QUEUE = None
MATLAB_RESULT_QUEUE = None
WORKER_ID = None


def set_matlab_context(task_q, result_q, worker_id):
    global MATLAB_TASK_QUEUE, MATLAB_RESULT_QUEUE, WORKER_ID
    MATLAB_TASK_QUEUE = task_q
    MATLAB_RESULT_QUEUE = result_q
    WORKER_ID = worker_id


def CBP_SPAN3_Mat_local(X):
    eng = matlab.engine.start_matlab()
    eng.cd(os.getcwd(), nargout=0)
    X = np.asarray(X, dtype=float)
    X_mat = matlab.double(X.tolist())
    results = []
    for row in X_mat:
        y_mat = eng.CBP_SPAN3_new(row, nargout=1)
        y = np.array(y_mat).reshape(-1)
        results.append(y)
    eng.quit()
    Y = np.vstack(results)
    Y = Y[:, :15]
    return Y

def CBP_SPAN3_Mat_remote(X):
    req = (WORKER_ID, np.asarray(X, dtype=float))
    MATLAB_TASK_QUEUE.put(req)
    Y = MATLAB_RESULT_QUEUE.get()
    return Y


def CBP_SPAN3_Mat(X):
    if MATLAB_TASK_QUEUE is None:
        return CBP_SPAN3_Mat_local(X)
    else:
        return CBP_SPAN3_Mat_remote(X)

def matlab_service(task_queue, result_queues):
    eng = matlab.engine.start_matlab()
    eng.cd(os.getcwd(), nargout=0)
    while True:
        task = task_queue.get()
        if task is None:
            break
        worker_id, X = task
        X = np.asarray(X, dtype=float)
        X_mat = matlab.double(X.tolist())
        results = []
        for row in X_mat:
            y_mat = eng.CBP_SPAN3_new(row, nargout=1)
            y = np.array(y_mat).reshape(-1)
            results.append(y)
        Y = np.vstack(results)
        Y = Y[:, :15]
        result_queues[worker_id].put(Y)
    eng.quit()

In [ ]:
# Distribution transform function

def normal_to_gumbel(U, mu, beta):
    U = np.asarray(U).ravel()

    p = norm.cdf(U)
    p = np.clip(p, 1e-15, 1 - 1e-15)

    return (mu - beta * np.log(-np.log(p))).ravel()


def normal_to_lognormal(U, mu_ln, sigma_ln):
    U = np.asarray(U).ravel()

    return np.exp(mu_ln + sigma_ln * U).ravel()


def predict_all_outputs(pce_models, X):
    cols = []

    for p in pce_models:
        cols.append(np.asarray(p.predict(X)).reshape(-1))

    return np.column_stack(cols)

In [ ]:
# PCE and input setting (physical and surrogate space)

# 1-3
D_mean = np.array([30.0, 10.0, 30.0])
D_std = 0.10 * D_mean

dist1 = Normal(loc=D_mean[0], scale=D_std[0])
dist2 = Normal(loc=D_mean[1], scale=D_std[1])
dist3 = Normal(loc=D_mean[2], scale=D_std[2])


# 4-6
L_mean = np.array([20.0, 10.0, 20.0])
L_std = 0.25 * L_mean

beta_L = (np.sqrt(6.0) / np.pi) * L_std
mu_L = L_mean - 0.5772 * beta_L

dist4 = Normal(0, 1)
dist5 = Normal(0, 1)
dist6 = Normal(0, 1)


# 7
m7_mean = 26.7
cov7 = 0.15

sigma_ln7 = np.sqrt(np.log(1.0 + cov7 ** 2))
mu_ln7 = np.log(m7_mean) - 0.5 * sigma_ln7 ** 2

dist7 = Normal(0, 1)


# 8
m8 = 435.0
s8 = 0.105 * m8

dist8 = Normal(m8, s8)


# Joint distribution
marg = [dist1, dist2, dist3, dist4, dist5, dist6, dist7, dist8]

joint = JointIndependent(marg)


# PCE basis
P = 8
polynomial_basis = TotalDegreeBasis(
    joint,
    P,
    hyperbolic=0.65
)


# Testing set and candidate pool
xx_testing = np.load("xx_testing.npy")
yy_testing = np.load("yy_testing.npy")
Xaptive = np.load("Xaptive.npy")

In [ ]:
# Theta criterion

def run(
    n_repeats: int = 10,
    cbp_func=None,
    init_n: int = 50,
    naddedsims: int = 450,
    batch_size: int = 10,
    nout: int = 15,
    n_test: int = 3000,
    test_seed: int = 123,
    show_progress: bool = True,
    store_float32: bool = True,
):

    if cbp_func is None:
        cbp_func = CBP_SPAN3_Mat

    # Fixed test subset
    rng = np.random.default_rng(int(test_seed))

    idx_test = rng.choice(
        xx_testing.shape[0],
        size=int(n_test),
        replace=False
    )

    X_test_sub = xx_testing[idx_test, :]

    
    # Reference variance
    mask_all = np.all(np.isfinite(yy_testing), axis=1)
    yy_testing_f = yy_testing[mask_all]

    rng_var = np.random.default_rng(int(test_seed) + 999)

    n_var = min(100000, yy_testing_f.shape[0])

    idx_var = rng_var.choice(
        yy_testing_f.shape[0],
        size=n_var,
        replace=False
    )

    var_true = np.var(
        yy_testing_f[idx_var],
        axis=0,
        ddof=1
    )

    results_all = []

    rep_iter = range(int(n_repeats))

    if show_progress:
        rep_iter = tqdm(rep_iter, desc="Repeats", leave=True)

    for rep in rep_iter:
        
        # Fixed candidate pool
        X_pool = Xaptive.copy()

        # Initial LHS samples
        lhs_init = LatinHypercubeSampling(
            distributions=marg,
            criterion=MaxiMin(metric=DistanceMetric.CHEBYSHEV),
            nsamples=int(init_n)
        )

        X_train = np.asarray(lhs_init._samples, dtype=float)

        # Transform to physical variables for Matlab evaluation
        X_train_phys = X_train.copy()

        for i in range(3):
            idx = 3 + i

            X_train_phys[:, idx] = normal_to_gumbel(
                X_train[:, idx],
                mu_L[i],
                beta_L[i]
            )

        X_train_phys[:, 6] = normal_to_lognormal(
            X_train[:, 6],
            mu_ln7,
            sigma_ln7
        )

        # Matlab model evaluation
        Y_train = cbp_func(X_train_phys)
        mask_rows = np.all(np.isfinite(Y_train), axis=1)
        X_train = X_train[mask_rows]
        Y_train = Y_train[mask_rows]

        # Initial PCE fitting
        least_squares = LeastSquareRegression()

        pce_list = []
        pceLAR_list = []

        for j in range(int(nout)):
            yj = Y_train[:, j]
            mask_y = np.isfinite(yj)

            if np.count_nonzero(mask_y) == 0:
                p = PolynomialChaosExpansion(
                    polynomial_basis=polynomial_basis,
                    regression_method=least_squares
                )

                p.fit(
                    X_train,
                    np.zeros(X_train.shape[0])
                )

                pce_list.append(p)
                pceLAR_list.append(p)

                continue

            p = PolynomialChaosExpansion(
                polynomial_basis=polynomial_basis,
                regression_method=least_squares
            )

            p.fit(
                X_train[mask_y],
                yj[mask_y]
            )

            try:
                pLAR = LeastAngleRegression.model_selection(p)
            except Exception:
                pLAR = p

            pce_list.append(p)
            pceLAR_list.append(pLAR)

        # Data storage
        Xadapted = X_train.copy()

        Yadapted = [
            Y_train[:, j].copy()
            for j in range(int(nout))
        ]

        hist_eval = []
        y_pred_hist = []
        eps_hist = []
        loo_hist = []

        total_to_add = int(naddedsims)
        bs = int(batch_size)

        step_iter = range(0, total_to_add, bs)

        if show_progress:
            step_iter = tqdm(
                step_iter,
                desc=f"AL batches run {rep + 1}",
                leave=False
            )

        for t in step_iter:
            n_pick = min(bs, total_to_add - t)

            if X_pool.shape[0] == 0:
                break
                
            # Theta sampling
            ThetaSampling = ThetaCriterionPCE(pceLAR_list)

            pos = ThetaSampling.run(
                existing_samples=Xadapted,
                candidate_samples=X_pool,
                nsamples=int(n_pick)
            )

            pos = np.asarray(pos, dtype=int)
            pos = np.unique(pos)

            newX = X_pool[pos, :]

            X_pool = np.delete(
                X_pool,
                np.sort(pos)[::-1],
                axis=0
            )

            newX_phys = newX.copy()

            for k in range(3):
                idx = 3 + k

                newX_phys[:, idx] = normal_to_gumbel(
                    newX[:, idx],
                    mu_L[k],
                    beta_L[k]
                )

            newX_phys[:, 6] = normal_to_lognormal(
                newX[:, 6],
                mu_ln7,
                sigma_ln7
            )

            newY = cbp_func(newX_phys)

            valid = np.all(np.isfinite(newY), axis=1)

            if np.count_nonzero(valid) == 0:
                continue

            newX = newX[valid, :]
            newY = newY[valid, :]

            Xadapted = np.vstack([
                Xadapted,
                newX
            ])

            for j in range(int(nout)):
                Yadapted[j] = np.r_[
                    Yadapted[j],
                    newY[:, j]
                ]

            for j in range(int(nout)):
                mask = np.isfinite(Yadapted[j])

                if np.count_nonzero(mask) == 0:
                    continue

                pce_list[j].fit(
                    Xadapted[mask],
                    Yadapted[j][mask]
                )

                try:
                    pceLAR_list[j] = LeastAngleRegression.model_selection(
                        pce_list[j]
                    )
                except Exception:
                    pceLAR_list[j] = pce_list[j]

            # LOO error
            loo_err_per = np.array([
                p.leaveoneout_error()
                for p in pceLAR_list
            ], dtype=float)

            
            # Variance error
            var_pce = []

            for p in pceLAR_list:
                try:
                    _, v = p.get_moments()
                except Exception:
                    v = np.nan

                var_pce.append(v)

            var_pce = np.array(var_pce, dtype=float)

            eps_per = np.abs(var_pce - var_true) / np.maximum(var_true, 1e-15)

            # Prediction value
            Y_pred_test = predict_all_outputs(
                pceLAR_list,
                X_test_sub
            )

            if store_float32:
                Y_pred_test = Y_pred_test.astype(np.float32)
                loo_err_per = loo_err_per.astype(np.float32)
                eps_per = eps_per.astype(np.float32)

            hist_eval.append(int(Xadapted.shape[0]))
            y_pred_hist.append(Y_pred_test)
            loo_hist.append(loo_err_per)
            eps_hist.append(eps_per)

        
        if len(y_pred_hist) > 0:
            Y_pred_hist = np.stack(y_pred_hist, axis=0)
        else:
            Y_pred_hist = np.empty(
                (0, int(n_test), int(nout))
            )

        eps_hist_arr = np.asarray(
            eps_hist,
            dtype=np.float32 if store_float32 else float
        )

        loo_hist_arr = np.asarray(
            loo_hist,
            dtype=np.float32 if store_float32 else float
        )

        if store_float32:
            Y_pred_hist = Y_pred_hist.astype(np.float32)

        results_all.append({
            "run": rep + 1,
            "idx_test": np.array(idx_test, dtype=int),
            "hist_eval": np.array(hist_eval, dtype=int),
            "Y_pred_hist": Y_pred_hist,
            "eps_hist": eps_hist_arr,
            "one_minus_Q2_hist": loo_hist_arr,
            "settings": {
                "init_n": int(init_n),
                "naddedsims": int(naddedsims),
                "batch_size": int(batch_size),
                "nout": int(nout),
                "n_test": int(n_test),
                "test_seed": int(test_seed),
                "store_float32": bool(store_float32),
                "theta_type": "normalized_multi_output_local_variance",
                "var_true_source": "finite yy_testing rows, random subset up to 100000, ddof=1",
            }
        })

    return results_all

In [ ]:
results = run(
    n_repeats=50,
    cbp_func=CBP_SPAN3_Mat,
    init_n=50,
    naddedsims=460,
    batch_size=10,
    nout=15,
    n_test=3000,
    test_seed=123,
    show_progress=True,
    store_float32=True,
)

with open("Theta.pkl", "wb") as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

print("done, total runs:", len(results))